# Neural Voice Identity Control & Deepfake Audio Analysis
**Disertație 2026 — Demo interactiv**

---

**⚠️ DISCLAIMER:**
Acest sistem este construit exclusiv în scop academic, pentru demonstrarea capacităților și limitelor
tehnologiilor de sinteză vocală și detecție deepfake audio. Vocile simulate sunt ale unor persoane publice
ale căror înregistrări sunt disponibile public. Sistemul **nu este** destinat înșelăciunii, uzului comercial
sau distribuției publice.

---

**Cum rulezi:**
1. Asigură-te că runtime-ul este T4 GPU (Runtime → Change runtime type)
2. Rulează toate celulele: Runtime → Run all
3. Accesează URL-ul Gradio generat la final

**Prerequisite:** Modelele fine-tuned (`.pth`) trebuie să existe în Google Drive.
Dacă nu ai antrenat încă, rulează `02_rvc_finetune.ipynb` prima dată.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU)"}')

In [ ]:
# Install dependencies
!pip install -q gradio>=4.7.0
!pip install -q librosa soundfile pydub
!pip install -q transformers accelerate
!pip install -q matplotlib plotly
!apt-get install -q -y ffmpeg
print('Dependencies installed.')

In [ ]:
# Setup RVC (clone if not present)
import os
import sys

RVC_DIR = '/content/RVC'

if not os.path.exists(RVC_DIR):
    print('Cloning RVC repository...')
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git {RVC_DIR} -q
    !pip install -q -r {RVC_DIR}/requirements.txt
    print('RVC cloned.')
else:
    print('RVC already present.')

sys.path.insert(0, RVC_DIR)

# Download HuBERT + RMVPE if needed
assets = [
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt',
     f'{RVC_DIR}/assets/hubert/hubert_base.pt'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt',
     f'{RVC_DIR}/assets/rmvpe/rmvpe.pt'),
]
for url, dest in assets:
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        print(f'Downloading {os.path.basename(dest)}...')
        !wget -q '{url}' -O '{dest}'
print('RVC setup complete.')

## Configurare

**TODO:** Completează `VOICE_CONFIG` cu numele reale ale vocilor după ce fine-tuning-ul e gata.

In [ ]:
from pathlib import Path

# ============================================================
# CONFIGURARE
# ============================================================
DRIVE_BASE = '/content/drive/MyDrive/disertatie'
MODELS_DIR = Path(f'{DRIVE_BASE}/models')

VOICE_CONFIG = {
    'voice_1': {
        'name': 'Călin Georgescu',
        'model_file': 'voice_1.pth',
        'index_file': 'voice_1.index',
        'transpose': 0,
    },
    'voice_2': {
        'name': 'Nicușor Dan',
        'model_file': 'voice_2.pth',
        'index_file': 'voice_2.index',
        'transpose': 0,
    },
    'voice_3': {
        'name': 'Diana Șoșoacă',
        'model_file': 'voice_3.pth',
        'index_file': 'voice_3.index',
        'transpose': 0,
    },
}

DEEPFAKE_MODEL_ID = 'MelodyMachine/Deepfake-audio-detection-V2'
# ============================================================

# Verificare ce modele există pe Drive
available_voices = {}
for vid, cfg in VOICE_CONFIG.items():
    model_path = MODELS_DIR / cfg['model_file']
    status = '✓' if model_path.exists() else '✗ (lipsește — rulează 02_rvc_finetune.ipynb)'
    print(f'{status} {cfg["name"]} ({vid})')
    if model_path.exists():
        available_voices[vid] = cfg['name']

if not available_voices:
    print('\n⚠ Niciun model găsit. App-ul pornește dar VC nu va funcționa.')
else:
    print(f'\nVoci disponibile: {list(available_voices.values())}')

In [ ]:
# Load deepfake detection model (only once)
import torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

print(f'Loading deepfake model: {DEEPFAKE_MODEL_ID}...')
_df_extractor = AutoFeatureExtractor.from_pretrained(DEEPFAKE_MODEL_ID)
_df_model = AutoModelForAudioClassification.from_pretrained(DEEPFAKE_MODEL_ID)
_df_model.eval()

# Determine which label index corresponds to "fake"
_fake_idx = next(
    (int(i) for i, lbl in _df_model.config.id2label.items()
     if any(k in lbl.lower() for k in ('fake', 'spoof', 'ai', 'synth', 'generated'))),
    1
)
print(f'Model loaded. Labels: {_df_model.config.id2label}')
print(f'Using label index {_fake_idx} as "fake"')

In [ ]:
# ============================================================
# Core function: detect_deepfake()
# ============================================================
import numpy as np
import librosa
import librosa.display
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import Tuple

SAMPLE_RATE_DF = 16000
WINDOW_S = 2.0
HOP_S = 0.5

def _predict_window(window: np.ndarray) -> float:
    inputs = _df_extractor(window, sampling_rate=SAMPLE_RATE_DF, return_tensors='pt', padding=True)
    with torch.no_grad():
        logits = _df_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).squeeze().numpy()
    return float(probs[_fake_idx])

def detect_deepfake(audio_path: str) -> Tuple[float, plt.Figure, plt.Figure]:
    audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE_DF, mono=True)
    win = int(WINDOW_S * SAMPLE_RATE_DF)
    hop = int(HOP_S * SAMPLE_RATE_DF)
    
    timestamps, scores = [], []
    for start in range(0, max(1, len(audio) - win + 1), hop):
        w = audio[start:start + win]
        if len(w) < win:
            w = np.pad(w, (0, win - len(w)))
        scores.append(_predict_window(w))
        timestamps.append((start + win // 2) / SAMPLE_RATE_DF)
    
    if not scores:
        padded = np.pad(audio, (0, win - len(audio)))
        scores = [_predict_window(padded)]
        timestamps = [len(audio) / (2 * SAMPLE_RATE_DF)]
    
    ts = np.array(timestamps)
    fs = np.array(scores)
    weights = np.hanning(len(fs)) + 0.1
    global_score = float(np.average(fs, weights=weights))
    
    # Timeline plot
    fig_tl, ax = plt.subplots(figsize=(10, 3))
    ax.fill_between(ts, fs, alpha=0.25, color='#e74c3c')
    ax.plot(ts, fs, color='#e74c3c', linewidth=1.8, label='Frame score')
    ax.axhline(y=global_score, color='#c0392b', linestyle='--', linewidth=1.5,
               label=f'Global: {global_score:.1%}')
    ax.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.7)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Timp (s)')
    ax.set_ylabel('P(AI-generat)')
    verdict = '🔴 AI-generat' if global_score > 0.5 else '🟢 Real'
    ax.set_title(f'Timeline Deepfake — {verdict} ({global_score:.1%})')
    ax.legend()
    fig_tl.tight_layout()
    
    # Spectrogram + overlay plot
    fig_sp, (ax_s, ax_p) = plt.subplots(2, 1, figsize=(12, 6),
                                          gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
    S = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE_DF, n_mels=128, fmax=8000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_dB, sr=SAMPLE_RATE_DF, x_axis='time', y_axis='mel',
                                    fmax=8000, ax=ax_s, cmap='magma')
    fig_sp.colorbar(img, ax=ax_s, format='%+2.0f dB')
    for t, score in zip(ts, fs):
        ax_s.axvspan(t - HOP_S / 2, t + HOP_S / 2, alpha=0.30,
                     color=plt.cm.RdYlGn_r(score), linewidth=0)
    ax_s.set_title('Mel Spectrogram cu Overlay (roșu = suspect AI)')
    ax_p.plot(ts, fs, color='#e74c3c', linewidth=1.5)
    ax_p.fill_between(ts, fs, alpha=0.25, color='#e74c3c')
    ax_p.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.7)
    ax_p.set_ylim(0, 1)
    ax_p.set_ylabel('P(fake)')
    ax_p.set_xlabel('Timp (s)')
    fig_sp.tight_layout()
    
    return global_score, fig_tl, fig_sp

print('detect_deepfake() defined.')

In [ ]:
# ============================================================
# Core function: voice_convert()
# ============================================================
import soundfile as sf
import tempfile

_rvc_instance = None
_rvc_current_voice = None

def voice_convert(
    audio_path: str,
    voice_id: str,
    transpose: int = 0,
    index_rate: float = 0.75,
    protect: float = 0.33,
) -> str:
    global _rvc_instance, _rvc_current_voice
    
    if voice_id not in VOICE_CONFIG:
        raise ValueError(f'Unknown voice: {voice_id}')
    
    cfg = VOICE_CONFIG[voice_id]
    model_path = str(MODELS_DIR / cfg['model_file'])
    index_path = str(MODELS_DIR / cfg['index_file'])
    
    if not Path(model_path).exists():
        raise FileNotFoundError(
            f'Model not found: {model_path}\n'
            'Run 02_rvc_finetune.ipynb to train the model first.'
        )
    
    # Load/reload VC only when voice changes (saves time)
    if _rvc_current_voice != voice_id:
        from infer.modules.vc.modules import VC
        from configs.config import Config
        _rvc_instance = VC(Config())
        _rvc_instance.get_vc(model_path)
        _rvc_current_voice = voice_id
        print(f'Loaded model: {cfg["name"]}')
    
    use_index = index_path if Path(index_path).exists() else ''
    pitch = transpose + cfg.get('transpose', 0)
    
    tgt_sr, output_audio = _rvc_instance.vc_single(
        sid=0,
        input_audio_path=audio_path,
        f0_up_key=pitch,
        f0_file=None,
        f0_method='rmvpe',
        file_index=use_index,
        index_rate=index_rate,
        filter_radius=3,
        resample_sr=0,
        rms_mix_rate=0.25,
        protect=protect,
    )
    
    tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    sf.write(tmp.name, output_audio, tgt_sr)
    return tmp.name

print('voice_convert() defined.')

In [ ]:
# ============================================================
# Gradio UI — Gradio callback functions
# ============================================================
import gradio as gr

VOICE_CHOICES = [cfg['name'] for cfg in VOICE_CONFIG.values()]
VOICE_ID_BY_NAME = {cfg['name']: vid for vid, cfg in VOICE_CONFIG.items()}


def vc_callback(audio_input, voice_name, transpose, index_rate, protect):
    """Gradio callback for Voice Conversion tab."""
    if audio_input is None:
        return None, '⚠ Niciun fișier audio. Uploadează sau înregistrează un .wav.'
    
    voice_id = VOICE_ID_BY_NAME.get(voice_name)
    if voice_id is None:
        return None, f'⚠ Voce necunoscută: {voice_name}'
    
    model_path = MODELS_DIR / VOICE_CONFIG[voice_id]['model_file']
    if not model_path.exists():
        return None, (
            f'⚠ Modelul pentru "{voice_name}" nu există încă.\n'
            'Rulează 02_rvc_finetune.ipynb pentru a antrena modelele.'
        )
    
    try:
        output_path = voice_convert(
            audio_input, voice_id,
            transpose=int(transpose),
            index_rate=float(index_rate),
            protect=float(protect),
        )
        return output_path, f'✓ Conversie completă → {voice_name}'
    except Exception as e:
        return None, f'✗ Eroare: {str(e)}'


def df_callback(audio_input):
    """Gradio callback for Deepfake Detection tab."""
    if audio_input is None:
        return None, None, '⚠ Niciun fișier audio.'
    
    try:
        score, fig_tl, fig_sp = detect_deepfake(audio_input)
        if score > 0.75:
            verdict = f'🔴 PROBABIL AI-GENERAT ({score:.1%})'
        elif score > 0.5:
            verdict = f'🟡 POSIBIL AI-GENERAT ({score:.1%})'
        elif score > 0.25:
            verdict = f'🟡 POSIBIL REAL ({score:.1%})'
        else:
            verdict = f'🟢 PROBABIL REAL ({score:.1%})'
        
        disclaimer = '\n⚠ Acuratețea pe limba română nu este validată științific.'
        return fig_tl, fig_sp, verdict + disclaimer
    except Exception as e:
        return None, None, f'✗ Eroare: {str(e)}'


print('Gradio callbacks defined.')

In [ ]:
# ============================================================
# Gradio UI — Layout
# ============================================================
import gradio as gr

ABOUT_TEXT = """
# Despre sistem

Acest sistem este proiectul practic al lucrării de disertație:
**"Neural Voice Identity Control and Deepfake Audio Analysis System"** (2026).

## Funcționalități

**Tab Voice Conversion:**
- Transformă timbrul vocal al unui fișier audio (upload sau microfon) în vocea uneia
  din cele 3 persoane publice românești pre-antrenate.
- Tehnologie: RVC v2 (Retrieval-based Voice Conversion), fine-tuned per voce.
- Limbaj-agnostic: funcționează pe orice text vorbit, nu doar română.

**Tab Deepfake Detection:**
- Analizează dacă un fișier audio este generat de AI sau real.
- Produce: scor global de probabilitate + grafic temporal + spectrogramă cu overlay.
- Tehnologie: model pre-antrenat wav2vec2 de pe HuggingFace (`MelodyMachine/Deepfake-audio-detection-V2`).

## ⚠️ Limitări cunoscute

- Calitatea vocii convertite depinde de cantitatea de date de antrenare și de calitatea audio input.
- Detectorul de deepfake este antrenat predominant pe **audio englezesc** — pe română poate
  produce rezultate mai puțin precise.
- Sistemul rulează pe Google Colab free tier (T4 GPU) — inferența poate dura 10-30s.

## ⚖️ Disclaimer legal și etic

Vocile simulate sunt ale unor persoane publice ale căror înregistrări sunt disponibile public
(interviuri, podcast-uri, discursuri). Sistemul este construit **exclusiv** în scop academic,
pentru demonstrarea capacităților și limitelor tehnologiilor de sinteză vocală și detecție deepfake.

**Utilizarea acestui sistem pentru înșelăciune, uzul comercial sau distribuție publică
este interzisă și poate constitui infracțiune.**
"""

with gr.Blocks(
    theme=gr.themes.Soft(),
    title='Neural Voice Identity Control & Deepfake Analysis',
) as app:
    gr.Markdown("""
    # 🎤 Neural Voice Identity Control & Deepfake Audio Analysis
    **Proiect academic — disertație 2026 | Nu pentru uz public sau comercial**
    """)

    with gr.Tabs():

        # ── TAB 1: VOICE CONVERSION ────────────────────────────────
        with gr.Tab('🎭 Voice Conversion'):
            gr.Markdown("""
            Uploadează un fișier audio (sau înregistrează de la microfon) și
            selectează vocea în care vrei să fie transformat.
            """)

            with gr.Row():
                with gr.Column(scale=1):
                    vc_input = gr.Audio(
                        label='Audio input',
                        sources=['upload', 'microphone'],
                        type='filepath',
                    )
                    vc_voice = gr.Dropdown(
                        label='Voce țintă',
                        choices=VOICE_CHOICES,
                        value=VOICE_CHOICES[0] if VOICE_CHOICES else None,
                    )
                    with gr.Accordion('Parametri avansați', open=False):
                        vc_transpose = gr.Slider(
                            label='Pitch shift (semitonuri)',
                            minimum=-12, maximum=12, step=1, value=0,
                        )
                        vc_index_rate = gr.Slider(
                            label='Index rate (retrieval weight)',
                            minimum=0.0, maximum=1.0, step=0.05, value=0.75,
                        )
                        vc_protect = gr.Slider(
                            label='Consonant protection',
                            minimum=0.0, maximum=0.5, step=0.01, value=0.33,
                        )
                    vc_btn = gr.Button('🎭 Convertește vocea', variant='primary')

                with gr.Column(scale=1):
                    vc_output = gr.Audio(label='Audio convertit', type='filepath')
                    vc_status = gr.Textbox(label='Status', interactive=False)

            vc_btn.click(
                fn=vc_callback,
                inputs=[vc_input, vc_voice, vc_transpose, vc_index_rate, vc_protect],
                outputs=[vc_output, vc_status],
            )

        # ── TAB 2: DEEPFAKE DETECTION ──────────────────────────────
        with gr.Tab('🔍 Deepfake Detection'):
            gr.Markdown("""
            Uploadează un fișier audio pentru a analiza dacă este generat de AI sau real.
            Sistemul produce un scor global și o analiză temporală.
            """)

            with gr.Row():
                with gr.Column(scale=1):
                    df_input = gr.Audio(
                        label='Audio de analizat',
                        sources=['upload', 'microphone'],
                        type='filepath',
                    )
                    df_btn = gr.Button('🔍 Analizează', variant='primary')
                    df_verdict = gr.Textbox(
                        label='Verdict',
                        interactive=False,
                        lines=3,
                    )

                with gr.Column(scale=2):
                    df_timeline = gr.Plot(label='Timeline probabilitate')
                    df_spectrogram = gr.Plot(label='Spectrogramă cu overlay')

            df_btn.click(
                fn=df_callback,
                inputs=[df_input],
                outputs=[df_timeline, df_spectrogram, df_verdict],
            )

        # ── TAB 3: ABOUT ───────────────────────────────────────────
        with gr.Tab('ℹ️ Despre / Disclaimer'):
            gr.Markdown(ABOUT_TEXT)

print('Gradio app defined. Run next cell to launch.')

In [ ]:
# ============================================================
# LANSARE APP
# Generează un URL public valabil 72h
# ============================================================
app.launch(
    share=True,          # Generează URL public gradio.live
    debug=False,
    quiet=False,
)